# Hàm hỗ trợ riêng cho bài toán phân cụm

Các hàm bổ sung nằm trong notebook này, không nối vào `ham_dung_chung.py`. File Python dùng chung được giữ nguyên byte theo ZIP cũ của nhóm. PC01–PC06 tự đọc và chạy các ô code định nghĩa ở đây trong kernel riêng, nên không cần chạy notebook này trước hay chia sẻ biến giữa các kernel.

Các ô chỉ định nghĩa hàm/lớp; không tự huấn luyện, xuất báo cáo hoặc chạy kiểm thử khi nạp. Run All notebook này cũng chỉ nạp định nghĩa. Muốn đọc luồng nghiệp vụ, bắt đầu ở PC01; muốn xem chi tiết triển khai, đọc các phần dưới.


In [1]:
# Chỉ đọc hàm gốc của dự án; không sửa ham_dung_chung.py.
from pathlib import Path
import sys
from functools import cache
from types import SimpleNamespace
sys.dont_write_bytecode = True
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'notebook/ham_dung_chung.py').is_file())
if str(ROOT/'notebook') not in sys.path:
    sys.path.insert(0,str(ROOT/'notebook'))
from ham_dung_chung import (PROJECT_ROOT, segmentation_study, segmentation_workflow,
    raw_fingerprints, read_model_parameters, segmentation_transform, rfm_features,
    save_model_parameters)


## 1. Phạm vi, mô hình, đối chứng và xuất kết quả tùy chọn

In [2]:
@cache
def phan_cum_tich_hop():
    """API phân cụm tích hợp, đặt trong file dùng chung theo cấu trúc gốc."""
    """Phân cụm tích hợp: tái sử dụng kiểm chứng cũ, mở rộng đối chứng và độ nhạy.

    Không sửa raw; mọi lượt mô hình có nhãn riêng và metadata truy vết.
    Dữ liệu tiền giữ Decimal trong RFM. Float chỉ dùng cho mô hình/biểu đồ.
    """
    from pathlib import Path
    from decimal import Decimal
    import hashlib
    import os
    import json
    import platform
    import re
    import time
    import numpy as np
    import pandas as pd
    import sklearn
    import matplotlib.pyplot as plt
    from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
    from sklearn.mixture import GaussianMixture
    from sklearn.preprocessing import StandardScaler
    from sklearn.neighbors import NearestNeighbors
    from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                                 calinski_harabasz_score, adjusted_rand_score,
                                 pairwise_distances)
    from threadpoolctl import threadpool_limits

    FEATURES = ['Recency', 'Frequency', 'Monetary']
    CUTOFF = pd.Timestamp('2022-12-31')
    SEED = 42

    def sha(path):
        with Path(path).open('rb') as stream:
            return hashlib.file_digest(stream, 'sha256').hexdigest()

    def write_json(path, value):
        Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2,
                                        default=str, allow_nan=False), encoding='utf-8')

    def root_from(start=None):
        start = Path(start or Path.cwd()).resolve()
        return next(p for p in [start, *start.parents]
                    if (p/'notebook/ham_dung_chung.py').is_file())

    def verify_inputs(root):
        """Hash common và nguồn theo bản kiểm chứng GD2 của lượt tích hợp."""
        root = Path(root)
        p = root/'data/processed/integration_inputs.json'
        if not p.exists():
            raise FileNotFoundError('Chạy GD2, sau đó gọi chuan_bi_phan_cum() đã nạp từ notebook hỗ trợ 00 để kiểm đầu vào.')
        expected = json.loads(p.read_text(encoding='utf-8'))
        for relative, digest in expected['files'].items():
            if sha(root/relative) != digest:
                raise ValueError('Nguồn/processed đã thay đổi, cần tái kiểm chứng: '+relative)
        return expected



    import tempfile
    import shutil
    from types import SimpleNamespace
    _model_cache = {}
    _result_name = 'ket_qua_phan_cum.csv'
    _parameter_name = 'mo_hinh_phan_cum.json'

    def model_directory(root,label):
        if re.fullmatch(r'[A-Za-z0-9_-]+',label) is None:
            raise ValueError('Tên lượt mô hình không hợp lệ')
        return Path(root)/'data/model/phan_khuc'/label

    def model_available(root,label):
        folder=model_directory(root,label)
        return (folder/_result_name).is_file() and (folder/_parameter_name).is_file()

    def _feature_stage(root,working):
        parent=segmentation_workflow().Workflow
        stage=parent.__new__(parent)
        stage.root=Path(root)
        stage.out=Path(working)
        stage.dataout=stage.out
        stage.manifest={}
        stage.audit()
        stage.prepare()
        stage.behavior()
        return stage

    def feature_data(root):
        """RFM/hành vi tính từ raw/common, không xuất CSV trung gian vào dự án."""
        with tempfile.TemporaryDirectory(prefix='da_rfm_') as directory:
            stage=_feature_stage(root,directory)
            return stage.features,stage.behaviors,stage.scope

    def _frozen_stage(root,working,metadata):
        stage=_feature_stage(root,working)
        selected=metadata['selected']
        stage.key=(selected['variant'],int(selected['k']))
        scaler=StandardScaler().fit(stage.features[FEATURES].astype(float))
        scaler.mean_=np.asarray(selected['scaler_mean'],dtype=float)
        scaler.scale_=np.asarray(selected['scaler_scale'],dtype=float)
        stage.scaler=scaler
        matrix=stage.features[FEATURES].astype(float).copy()
        if stage.key[0]=='TN2':matrix[['Frequency','Monetary']]=np.log1p(matrix[['Frequency','Monetary']])
        stage.x=scaler.transform(matrix)
        centers=np.asarray(selected['center_coordinates'],dtype=float)
        def predict(values):
            return ((np.asarray(values)[:,None,:]-centers[None,:,:])**2).sum(axis=2).argmin(axis=1)
        labels=predict(stage.x)
        stage.model=SimpleNamespace(labels_=labels,cluster_centers_=centers,predict=predict)
        stage.idx=np.sort(np.random.default_rng(SEED).choice(len(labels),min(5000,len(labels)),replace=False))
        return stage

    def open_model(root,label):
        """Đọc CSV kết quả và JSON tham số; dựng bảng phân tích tạm khi cần."""
        key=(str(Path(root).resolve()),label)
        if key in _model_cache:return Path(_model_cache[key].name)
        verify_inputs(root)
        folder=model_directory(root,label)
        if not model_available(root,label):
            raise FileNotFoundError('Chưa có CSV kết quả và JSON mô hình; chạy PC03 để huấn luyện.')
        metadata=json.loads((folder/_parameter_name).read_text(encoding='utf-8'))
        if metadata['state']!='completed' or metadata['raw_sha256']!=raw_fingerprints(root):
            raise ValueError('Mô hình khác nguồn hoặc chưa hoàn tất')
        if metadata.get('reference_date')!='2022-12-31' or metadata.get('status_scope')!=['delivered']:
            raise ValueError('Mô hình khác chính sách ngày/trạng thái của nghiên cứu')
        if sha(folder/_result_name)!=metadata['result_sha256']:
            raise ValueError('CSV kết quả đã thay đổi, cần kiểm chứng lại thay vì dùng ngầm.')
        temporary=tempfile.TemporaryDirectory(prefix='da_phan_cum_view_')
        try:
            stage=_frozen_stage(root,temporary.name,metadata)
            selected=metadata['selected']
            predicted=np.array([int(selected['center_to_cluster'][str(int(v))]) for v in stage.model.labels_])
            result=pd.read_csv(folder/_result_name,dtype={'customer_id':str,'Monetary':str})
            expected=stage.features[['customer_id','Recency','Frequency','Monetary']].merge(
                result,on='customer_id',validate='one_to_one',suffixes=('_source','_result'))
            assert len(expected)==len(result)==len(stage.features)
            for column in ['Recency','Frequency']:
                assert expected[column+'_source'].eq(expected[column+'_result']).all()
            assert expected.Monetary_source.eq(expected.Monetary_result.map(Decimal)).all()
            ordered=result.set_index('customer_id').loc[stage.features.customer_id]
            np.testing.assert_array_equal(predicted,ordered.cluster.to_numpy())
            stage.profile()
            assert stage.mapping=={int(k):int(v) for k,v in selected['center_to_cluster'].items()}
            stage.save('silhouette_sample',stage.features.iloc[stage.idx][['customer_id']])
            save_model_parameters(stage.out,{**selected,'center_to_cluster':{int(k):int(v) for k,v in selected['center_to_cluster'].items()}})
            write_json(stage.out/'integration_manifest.json',metadata)
            shutil.copyfile(folder/_result_name,stage.out/_result_name)
        except Exception:
            temporary.cleanup()
            raise
        _model_cache[key]=temporary
        return Path(temporary.name)

    def _names_from_profile(frame):
        ranked=frame.groupby('cluster').Monetary.median().sort_values().index
        if len(ranked)==3:
            names=['Lâu chưa mua, giá trị tích lũy thấp','Mức hoạt động và giá trị trung gian','Mua nhiều, giá trị tích lũy cao']
        else:names=['Nhóm '+str(int(v)) for v in ranked]
        return dict(zip(ranked,names))

    def commit_model(root,label,working):
        """Chỉ lưu kết quả khách hàng và tham số mô hình đã chọn — đúng hai file."""
        working=Path(working)
        metadata=json.loads((working/'integration_manifest.json').read_text(encoding='utf-8'))
        features=pd.read_csv(working/'rfm_exact.csv',dtype={'customer_id':str,'Monetary':str})
        members=pd.read_csv(working/'customer_segments.csv',dtype={'customer_id':str})
        result=features[['customer_id','Recency','Frequency','Monetary']].merge(
            members[['customer_id','cluster']],on='customer_id',validate='one_to_one')
        result['segment']=result.cluster.map(_names_from_profile(members))
        if (working/_result_name).exists():
            proposed=pd.read_csv(working/_result_name,dtype={'customer_id':str})
            if set(proposed.customer_id)==set(result.customer_id) and proposed.customer_id.is_unique:
                result['segment']=result.customer_id.map(proposed.set_index('customer_id').segment)
        folder=model_directory(root,label)
        folder.mkdir(parents=True,exist_ok=True)
        fd,csv_temp=tempfile.mkstemp(prefix='.result_',suffix='.tmp',dir=folder);os.close(fd)
        fd,json_temp=tempfile.mkstemp(prefix='.model_',suffix='.tmp',dir=folder);os.close(fd)
        try:
            result.to_csv(csv_temp,index=False)
            metadata.update(storage_format='selected-rfm-model-json-v1',features=FEATURES,
                            result_file=_result_name,result_sha256=sha(csv_temp))
            write_json(json_temp,metadata)
            os.replace(csv_temp,folder/_result_name)
            os.replace(json_temp,folder/_parameter_name)
        finally:
            Path(csv_temp).unlink(missing_ok=True);Path(json_temp).unlink(missing_ok=True)
        return result

    def store_final(root,label,frame):
        """CSV phân cụm là đầu ra chính; chỉ cập nhật nếu nhãn/tên thật sự thay đổi."""
        folder=model_directory(root,label)
        old=pd.read_csv(folder/_result_name,dtype={'customer_id':str})
        if old.equals(frame.reset_index(drop=True)):return
        base=open_model(root,label)
        frame.to_csv(base/_result_name,index=False)
        commit_model(root,label,base)

    def ensure_main_experiments(root,label):
        """Tính lại bảng 14 cấu hình khi mở PC03; chỉ giữ tạm trong kernel."""
        base=open_model(root,label)
        if (base/'experiment_grid.csv').exists():return
        parent=segmentation_workflow().Workflow
        stage=parent.__new__(parent)
        stage.root=Path(root);stage.out=base;stage.dataout=base;stage.manifest={}
        stage.features=pd.read_csv(base/'rfm_exact.csv',dtype={'customer_id':str,'Monetary':str})
        stage.features['Monetary']=stage.features.Monetary.map(Decimal)
        stage.experiments();stage.stability();stage.hierarchy()
        selected=json.loads((base/'integration_manifest.json').read_text())['selected']
        key=(selected['variant'],int(selected['k']))
        rows=[]
        with threadpool_limits(limits=1):
            x=stage.matrices[key[0]]
            original=pd.read_csv(base/'customer_segments.csv').cluster.to_numpy()
            for seed in [7,21,84]:
                labels=KMeans(n_clusters=key[1],n_init=10,max_iter=300,random_state=seed).fit_predict(x)
                rows.append({'seed':seed,'ari_full':adjusted_rand_score(original,labels)})
        pd.DataFrame(rows).to_csv(base/'selected_full_seed_stability.csv',index=False)
        np.testing.assert_equal(adjusted_rand_score(original,stage.models[key].labels_),1.)

    def ensure_status_sensitivity(root,label):
        base=open_model(root,label)
        if (base/'status_sensitivity.csv').exists():return
        metadata=json.loads((base/'integration_manifest.json').read_text())
        stage=_frozen_stage(root,base,metadata)
        stage.sensitivity()

    class Study(segmentation_study().Study):
        """Giữ API notebook cũ; thêm xuất bảng, biểu đồ và kiểm chứng nguồn."""
        def __init__(self, root, part, baseline=None):
            self.root=Path(root)
            self.input_evidence=verify_inputs(root)
            self.part=part
            self.model_root=self.root/'data/model/phan_khuc'
            self.records=[]
            self.figure_count=0
            self.raw_before=raw_fingerprints(self.root)
            self.baseline_label=baseline or os.environ.get('DATATHON_FEATURE_RUN','combined_v1')
            self.model_label=self.baseline_label
            if model_available(self.root,self.baseline_label):
                self.base=open_model(self.root,self.baseline_label)
                state=json.loads((self.base/'integration_manifest.json').read_text(encoding='utf-8'))
                if state['state']!='completed' or state['raw_sha256']!=self.raw_before:
                    raise ValueError('Gói đầu vào chưa hoàn tất hoặc khác nguồn hiện tại')
                self.data=self.base
                self.manifest=read_model_parameters(self.base)
                self.f=pd.read_csv(self.base/'rfm_exact.csv',dtype={'customer_id':str,'Monetary':str},parse_dates=['first_order_date','last_order_date'])
                self.f['Monetary']=self.f.Monetary.map(Decimal)
                self.behavior=pd.read_csv(self.base/'customer_behavior.csv',dtype={'customer_id':str})
                self.scope=pd.read_csv(self.base/'customer_scope.csv',dtype={'customer_id':str})
            else:
                self.f,self.behavior,self.scope=feature_data(self.root)
                self.base=self.data=None
                self.manifest={}
            self.report_root=self.root/'reports/phan_khuc/tich_hop'
            self.out=self.report_root/part
            self.export_reports=os.environ.get('DATATHON_EXPORT_REPORTS','0')=='1'
            if self.export_reports:self.out.mkdir(parents=True,exist_ok=True)

        def model_files(self,label=None):
            return open_model(self.root,label or self.model_label)

        def use_model(self,label):
            base=open_model(self.root,label)
            state=json.loads((base/'integration_manifest.json').read_text(encoding='utf-8'))
            if state['state']!='completed' or state['raw_sha256']!=raw_fingerprints(self.root):
                raise ValueError('Lượt mô hình chưa hoàn tất hoặc khác nguồn')
            self.model_label=label
            self.modelbase=self.modeldata=base
            self.modelmanifest=read_model_parameters(base)
            self.a=pd.read_csv(base/'customer_segments.csv',dtype={'customer_id':str})
            if set(self.a.customer_id)!=set(self.f.customer_id):
                raise ValueError('Mô hình khác tập khách RFM')
            return self.modelmanifest

        def table(self, name, title, frame):
            if self.export_reports:
                frame.to_csv(self.out/(name+'.csv'), index=False)
            return super().table(name, title, frame)

        def fig(self, name, title, figure):
            from io import BytesIO
            from IPython.display import Image, display

            figure.tight_layout()
            if self.export_reports:
                figure.savefig(self.out/(name+'.png'), dpi=130, bbox_inches='tight')
            # Hiển thị PNG cả khi hàm dùng chung chọn backend Agg.
            with BytesIO() as image:
                figure.savefig(image, format='png', dpi=130, bbox_inches='tight')
                display(Image(data=image.getvalue()))
            plt.close(figure)
            self.figure_count += 1

        def finish(self):
            result = super().finish()
            if self.export_reports:
                write_json(self.out/'execution.json', result)
            return result

    class Workflow(segmentation_workflow().Workflow):
        """Một lượt huấn luyện chính thực sự; giữ bộ kiểm tra RFM, trạng thái cũ."""
        def __init__(self, root=None, label='combined_v1'):
            self.root=root_from(root)
            verify_inputs(self.root)
            if model_available(self.root,label):
                raise FileExistsError('Lượt mô hình đã tồn tại; chọn nhãn mới để huấn luyện.')
            model_directory(self.root,label)  # validate label before creating temporary storage
            self.label=label
            temporary=tempfile.TemporaryDirectory(prefix='da_model_train_')
            _model_cache[(str(self.root.resolve()),label)]=temporary
            self.out=Path(temporary.name)
            self.dataout=self.out
            self.raw_before=raw_fingerprints(self.root)
            self.manifest={}
            self.save_json('integration_manifest',{'state':'running','label':label})

        def save_json(self, name, value):
            write_json(self.out/(name+'.json'), value)

        def finish(self):
            # Kiểm seed trên toàn tập cho chính mô hình được chọn, không chỉ mô hình mẫu.
            selected_rows = self.stability_table.loc[
                self.stability_table.variant.eq(self.key[0]) & self.stability_table.k.eq(self.key[1]),
                ['seed','ari']].rename(columns={'ari':'ari_full'})
            self.save('selected_full_seed_stability', selected_rows)
            super().finish()
            self.save_json('audit_findings', self.audit_info)
            self.save_json('feature_quality', self.quality)
            agreements = pd.read_csv(self.out/'hierarchy_agreement.csv')
            self.save_json('hierarchy_agreement', {str(int(row.iloc[0])):float(row.iloc[1]) for _,row in agreements.iterrows()})
            exact = str(sum(self.features.Monetary, Decimal(0)))
            manifest = {'state':'completed','label':self.label,
                        'reference_date':'2022-12-31','status_scope':['delivered'],
                        'selected':self.manifest['selected'], 'exact_net_total':exact,
                        'raw_sha256':raw_fingerprints(self.root),
                        'processed_sha256':{p.name:sha(p) for p in (self.root/'data/processed').glob('*common.csv')},
                        'versions':{'python':platform.python_version(),'numpy':np.__version__,
                                    'pandas':pd.__version__,'sklearn':sklearn.__version__},
                        'selection_rule':'14 TN1/TN2 configurations; minimum share 1%; rank S+DB; first top3 candidate mean seed ARI >=.90. Business decision remains exploratory.',
                        'metric_scope':'Full model: silhouette fixed5000, DB/inertia full. Countermodels: same independent sample scaler and 5000 rows, TN1 only.',
                        'limitations':['No status event history/verified extraction time','Monetary is delivered net, not settled cash/profit/CLV','No temporal validation or demonstrated intervention effect']}
            self.save_json('integration_manifest',manifest)
            self.manifest['limitations'] = manifest['limitations']
            commit_model(self.root,self.label,self.out)
            return self.manifest

    def independent_audit(root):
        """Đối soát Decimal từ dòng hàng raw độc lập với bảng common và RFM."""
        root = Path(root)
        evidence = verify_inputs(root)
        audit_path = root/'data/processed/independent_rfm_audit.json'
        reference_path = open_model(root,'combined_v1')/'rfm_exact.csv' if model_available(root,'combined_v1') else None
        fingerprint = hashlib.sha256(json.dumps(evidence['files'],sort_keys=True).encode()).hexdigest() + (sha(reference_path) if reference_path else 'fresh_inputs')
        if audit_path.exists():
            cached = json.loads(audit_path.read_text(encoding='utf-8'))
            if cached.get('input_fingerprint') == fingerprint and cached.get('reference') == 'current_source_verified_model':
                return pd.DataFrame([cached])
        from collections import defaultdict
        raw = root/'data/raw'
        items = pd.read_csv(raw/'order_items.csv',dtype=str,keep_default_na=False,
                            usecols=['order_id','quantity','unit_price','discount_amount'])
        totals = defaultdict(lambda:Decimal(0))
        for order, qty, price, discount in items.itertuples(index=False,name=None):
            totals[order] += Decimal(qty)*Decimal(price)-Decimal(discount)
        common = pd.read_csv(root/'data/processed/orders_common.csv', dtype=str,
                             keep_default_na=False,usecols=['order_id','customer_id','order_date_parsed','order_status','net_amount'])
        orders = pd.read_csv(raw/'orders.csv',dtype=str,keep_default_na=False,
                             usecols=['order_id','customer_id','order_date','order_status'])
        if not orders.order_id.is_unique or not common.order_id.is_unique:
            raise ValueError('Duplicate order keys')
        joined = common.merge(orders,on='order_id',validate='one_to_one',suffixes=('_common','_raw'))
        assert len(joined)==len(common)==len(orders)
        assert joined.customer_id_common.eq(joined.customer_id_raw).all()
        assert joined.order_status_common.eq(joined.order_status_raw).all()
        assert pd.to_datetime(joined.order_date_parsed).eq(pd.to_datetime(joined.order_date)).all()
        assert common.net_amount.map(Decimal).eq(common.order_id.map(totals)).all()
        selected = common.loc[common.order_status.eq('delivered')].copy()
        selected['order_date'] = pd.to_datetime(selected.order_date_parsed)
        selected = selected.loc[selected.order_date.le(CUTOFF)]
        selected['net_amount'] = selected.net_amount.map(Decimal)
        fresh = rfm_features().rfm(selected)
        if reference_path is not None:
            baseline=pd.read_csv(reference_path,dtype={'customer_id':str,'Monetary':str})
            baseline['Monetary']=baseline.Monetary.map(Decimal)
            check=fresh.merge(baseline,on='customer_id',validate='one_to_one',suffixes=('_fresh','_old'))
            assert len(check)==len(fresh)==len(baseline)
            assert all(check[f+'_fresh'].eq(check[f+'_old']).all() for f in FEATURES)
        compared=reference_path is not None
        result = {'input_fingerprint':fingerprint,'orders':len(orders),'eligible_orders':len(selected),'customers':len(fresh),
                  'exact_net_total':str(sum(fresh.Monetary,Decimal(0))),
                  'all_order_net_exact_match_raw':True,'all_customer_rfm_exact_match_baseline':compared,'reference':'current_source_verified_model' if compared else 'fresh_raw_common_check',
                  'method':'Independent Decimal sum from every raw item; one-to-one order keys and source date/status/customer equality; exact per-customer current-model RFM when available.'}
        write_json(root/'data/processed/independent_rfm_audit.json',result)
        return pd.DataFrame([result])

    def estimator(name, params, seed=42, linkage='ward'):
        if name=='K-Means':
            return KMeans(n_clusters=int(params['k']),n_init=10,max_iter=300,random_state=seed)
        if name=='Hierarchical':
            return AgglomerativeClustering(n_clusters=int(params['k']),linkage=linkage)
        if name=='GMM':
            return GaussianMixture(n_components=int(params['k']),covariance_type='full',n_init=3,max_iter=300,random_state=seed)
        if name=='DBSCAN':
            return DBSCAN(eps=float(params['eps']),min_samples=int(params['min_samples']))
        raise ValueError(name)

    def cluster_info(labels):
        valid = labels != -1
        counts = np.unique(labels[valid],return_counts=True)[1]
        return {'clusters':len(counts),'coverage':float(valid.mean()),
                'min_cluster_share_all':float(counts.min()/len(labels)) if len(counts) else 0.,
                'max_cluster_share_all':float(counts.max()/len(labels)) if len(counts) else 0.,
                'noise_share':float((~valid).mean())}

    def metric_row(x, labels, distances=None):
        row = cluster_info(labels)
        valid = labels != -1
        row.update(silhouette=np.nan,davies_bouldin=np.nan,calinski_harabasz=np.nan)
        if 2 <= row['clusters'] < int(valid.sum()):
            row['silhouette'] = float(silhouette_score(
                distances[np.ix_(valid,valid)] if distances is not None else x[valid],
                labels[valid],metric='precomputed' if distances is not None else 'euclidean'))
            row['davies_bouldin'] = float(davies_bouldin_score(x[valid],labels[valid]))
            row['calinski_harabasz'] = float(calinski_harabasz_score(x[valid],labels[valid]))
        return row

    def compare_models(root, label):
        """36 cấu hình cùng 5000 khách, TN1, scaler mẫu; DBSCAN có cửa sàng lọc rõ."""
        root=Path(root);base=open_model(root,label)
        dest=base/'doi_chung';dest.mkdir(exist_ok=True)
        fingerprint=sha(base/'rfm_exact.csv')
        if (dest/'metadata.json').exists():
            meta=json.loads((dest/'metadata.json').read_text())
            if meta['rfm_sha256']!=fingerprint:raise ValueError('Cache so sánh khác RFM')
            return pd.read_csv(dest/'scores.csv')
        f=pd.read_csv(base/'rfm_exact.csv',dtype={'customer_id':str})
        idx=np.random.default_rng(42).choice(len(f),min(5000,len(f)),replace=False)
        sample=f.iloc[idx].reset_index(drop=True)
        scaler=StandardScaler();x=scaler.fit_transform(sample[FEATURES].astype(float))
        sample.to_csv(dest/'sample_rfm.csv',index=False)
        rows=[];all_labels={};configs=[]
        for k in range(2,9):
            for name in ['K-Means','Hierarchical','GMM']:configs.append((name,{'k':k}))
        with threadpool_limits(limits=1):
            for min_samples in [5,10,20]:
                dist=NearestNeighbors(n_neighbors=min_samples).fit(x).kneighbors(x)[0][:,-1]
                for q in [.5,.75,.9,.95,.99]:
                    eps=float(np.quantile(dist,q))
                    if eps>0:configs.append(('DBSCAN',{'eps':eps,'min_samples':min_samples,'quantile':q}))
            distances=pairwise_distances(x)
            np.fill_diagonal(distances,0)
            for run_id,(name,params) in enumerate(configs):
                started=time.monotonic();model=estimator(name,params);labels=model.fit_predict(x)
                row={'run_id':run_id,'algorithm':name,'params':json.dumps(params),
                     'seconds':time.monotonic()-started,**metric_row(x,labels,distances)}
                row['converged']=bool(getattr(model,'converged_',True))
                row['eligible']=(2<=row['clusters']<=8 and row['coverage']>=.95
                                 and row['min_cluster_share_all']>=.02 and row['converged'])
                rows.append(row);all_labels[str(run_id)]=labels
                print('Đối chứng',name,params,'S=',round(row['silhouette'],4),flush=True)
        scores=pd.DataFrame(rows);scores.to_csv(dest/'scores.csv',index=False)
        np.savez_compressed(dest/'labels.npz',**all_labels)
        # Không dùng fallback không đạt để công bố người thắng.
        selected=[];diagnostics=[]
        for name,g in scores.groupby('algorithm',sort=False):
            eligible=g.loc[g.eligible & g.silhouette.notna()]
            pool=eligible if len(eligible) else g.loc[g.silhouette.notna()]
            if len(pool):
                row=pool.sort_values('silhouette',ascending=False).iloc[0]
                (selected if len(eligible) else diagnostics).append(row)
        pd.DataFrame(selected).to_csv(dest/'representatives_eligible.csv',index=False)
        pd.DataFrame(diagnostics,columns=scores.columns).to_csv(dest/'diagnostics_ineligible.csv',index=False)
        write_json(dest/'metadata.json',{'rfm_sha256':fingerprint,'sample_customers':len(sample),
            'sample_indices':idx.tolist(),'features':FEATURES,'variant':'TN1','seed':42,
            'scaler_mean':scaler.mean_.tolist(),'scaler_scale':scaler.scale_.tolist(),
            'sample_scaler_fit_independently':True,'dbscan_metrics':'Non-noise subset; not same population as full coverage algorithms',
            'selection':'Eligible only:2..8 clusters,coverage>=95%,min share>=2%,GMM converged. Ineligible models shown separately, never winners.'})
        return scores

    def sensitivity_models(root,label):
        """Độ ổn định trên mẫu: seed,80%,P99,tham số; tách ARI nhiễu DBSCAN."""
        compare_models(root,label)
        base=open_model(root,label)/'doi_chung'
        if (base/'sensitivity.csv').exists():return pd.read_csv(base/'sensitivity.csv')
        f=pd.read_csv(base/'sample_rfm.csv',dtype={'customer_id':str})
        meta=json.loads((base/'metadata.json').read_text())
        x=(f[FEATURES].to_numpy(float)-np.array(meta['scaler_mean']))/np.array(meta['scaler_scale'])
        scores=pd.read_csv(base/'scores.csv');saved=np.load(base/'labels.npz')
        refs={}
        for name in ['K-Means','Hierarchical','GMM']:
            row=scores.loc[scores.algorithm.eq(name)&scores.params.map(lambda s:json.loads(s).get('k')==3)].iloc[0]
            refs[name]=row
        db=scores.loc[scores.algorithm.eq('DBSCAN')&scores.silhouette.notna()]
        if len(db):refs['DBSCAN']=db.sort_values('silhouette',ascending=False).iloc[0]
        rows=[]
        def compare(name,row,case,matrix,indices=None,seed=42,params=None,linkage='ward'):
            ref=saved[str(int(row.run_id))]
            ref=ref if indices is None else ref[indices]
            labels=estimator(name,params or json.loads(row.params),seed,linkage).fit_predict(matrix)
            valid=(ref!=-1)&(labels!=-1)
            ari_nonnoise=adjusted_rand_score(ref[valid],labels[valid]) if valid.sum()>1 and len(np.unique(ref[valid]))>1 and len(np.unique(labels[valid]))>1 else np.nan
            rows.append({'algorithm':name,'case':case,'seed':seed,
                         'ari_all_including_noise':adjusted_rand_score(ref,labels),
                         'ari_common_nonnoise':ari_nonnoise,'common_nonnoise':int(valid.sum()),
                         'reference_eligible':bool(row.eligible),'comparison_customers':len(labels),**cluster_info(labels)})
        with threadpool_limits(limits=1):
            for name,row in refs.items():
                if name in ['K-Means','GMM']:
                    for seed in [0,7,21,42,99]:compare(name,row,'seed',x,seed=seed)
                for seed in [7,21,99]:
                    idx=np.random.default_rng(seed).choice(len(x),int(.8*len(x)),replace=False)
                    compare(name,row,'subset80_fixed_scaler_'+str(seed),x[idx],indices=idx)
                changed=f[FEATURES].astype(float).copy()
                changed[['Frequency','Monetary']]=changed[['Frequency','Monetary']].clip(upper=changed[['Frequency','Monetary']].quantile(.99),axis=1)
                compare(name,row,'P99_refit_scaler_and_model',StandardScaler().fit_transform(changed))
                if name=='Hierarchical':
                    for linkage in ['complete','average']:compare(name,row,'linkage_'+linkage,x,linkage=linkage)
                if name=='DBSCAN':
                    params=json.loads(row.params)
                    for factor in [.9,1.1]:compare(name,row,'eps_x'+str(factor),x,params={**params,'eps':params['eps']*factor})
        result=pd.DataFrame(rows);result.to_csv(base/'sensitivity.csv',index=False)
        with threadpool_limits(limits=1):
            km=estimator('K-Means',{'k':3}).fit(x)
        d=np.sort(km.transform(x),axis=1);margin=(d[:,1]-d[:,0])/(d[:,1]+1e-12)
        border=f.assign(relative_margin=margin,selected_bottom5pct=margin<=np.quantile(margin,.05))
        border.to_csv(base/'boundary_diagnostics.csv',index=False)
        return result

    def plot_comparison(root, label):
        """PCA chỉ để hiển thị; ba thuật toán cùng k3 và DBSCAN chẩn đoán riêng."""
        from sklearn.decomposition import PCA
        base=open_model(root,label)/'doi_chung'
        f=pd.read_csv(base/'sample_rfm.csv')
        meta=json.loads((base/'metadata.json').read_text())
        x=(f[FEATURES].to_numpy(float)-meta['scaler_mean'])/meta['scaler_scale']
        pca=PCA(n_components=2);points=pca.fit_transform(x)
        scores=pd.read_csv(base/'scores.csv');saved=np.load(base/'labels.npz')
        fig,axes=plt.subplots(2,2,figsize=(12,9),sharex=True,sharey=True)
        for ax,name in zip(axes.flat,['K-Means','Hierarchical','GMM','DBSCAN']):
            pool=scores.loc[scores.algorithm.eq(name)]
            if name!='DBSCAN':pool=pool.loc[pool.params.map(lambda s:json.loads(s).get('k')==3)]
            row=pool.dropna(subset=['silhouette']).sort_values('silhouette',ascending=False).iloc[0]
            labels=saved[str(int(row.run_id))]
            ranking=f.assign(label=labels).loc[labels!=-1].groupby('label').Monetary.median().sort_values()
            for index,cluster in enumerate(ranking.index):
                mask=labels==cluster
                color=plt.get_cmap('viridis')((index+1)/(len(ranking)+1))
                ax.scatter(points[mask,0],points[mask,1],s=7,alpha=.35,color=color,label='Cụm '+str(cluster))
                center=points[mask].mean(axis=0)
                ax.scatter(*center,marker='X',s=65,color='red',edgecolor='black',linewidth=.5)
            if (labels==-1).any():ax.scatter(points[labels==-1,0],points[labels==-1,1],s=12,c='gray',marker='x',label='Nhiễu')
            suffix=' — chẩn đoán, không đạt' if not bool(row.eligible) else ''
            ax.set_title(name+' / '+str(int(row.clusters))+' cụm'+suffix,fontsize=10)
            ax.set_xlabel('PC1');ax.set_ylabel('PC2');ax.legend(fontsize=7)
        fig.suptitle(f'PCA mẫu TN1: {pca.explained_variance_ratio_.sum():.2%} phương sai; X đỏ là trung bình điểm trong cụm',fontsize=11)
        return fig

    def plot_sensitivity(root,label):
        base=open_model(root,label)/'doi_chung'
        d=pd.read_csv(base/'sensitivity.csv')
        sampling=d.loc[d['case'].str.startswith('subset80')].groupby('algorithm').ari_all_including_noise.min()
        p99=d.loc[d['case'].eq('P99_refit_scaler_and_model')].set_index('algorithm').ari_all_including_noise
        table=pd.DataFrame({'ARI mẫu 80% thấp nhất':sampling,'ARI P99':p99})
        fig,ax=plt.subplots(figsize=(10,4));table.plot.bar(ax=ax,rot=0)
        ax.set_ylabel('ARI gồm nhãn nhiễu');ax.set_xlabel('Thuật toán')
        ax.set_title('Đọc cùng số cụm/bao phủ; ARI cao không tự chứng minh tốt')
        ax.set_ylim(min(0,float(table.min().min())-.05),1.05)
        return fig

    return SimpleNamespace(
        ensure_main_experiments=ensure_main_experiments,
        ensure_status_sensitivity=ensure_status_sensitivity,
        model_available=model_available,
        open_model=open_model,
        commit_model=commit_model,
        store_final=store_final,
        feature_data=feature_data,
        Study=Study,
        Workflow=Workflow,
        verify_inputs=verify_inputs,
        independent_audit=independent_audit,
        compare_models=compare_models,
        sensitivity_models=sensitivity_models,
        plot_comparison=plot_comparison,
        plot_sensitivity=plot_sensitivity,
        cluster_info=cluster_info,
        metric_row=metric_row,
        estimator=estimator,
        sha=sha,
        write_json=write_json,
        root_from=root_from,
    )



## 2. Bộ kiểm thử phân cụm — chỉ chạy khi chủ động gọi

In [3]:
def kiem_thu_phan_cum():
    """Chạy 26 kiểm thử sau khi đã Run All và Save PC01–PC06; không xuất báo cáo."""
    """Kiểm thử thuật toán nền và nghiệm thu đầu ra thật của bản tích hợp."""
    from pathlib import Path
    from decimal import Decimal
    import json
    import sys
    import unittest
    import numpy as np
    import pandas as pd
    from sklearn.metrics import adjusted_rand_score
    ROOT=PROJECT_ROOT
    sys.path.insert(0,str(ROOT/'notebook'))
    cluster_info = phan_cum_tich_hop().cluster_info
    metric_row = phan_cum_tich_hop().metric_row
    estimator = phan_cum_tich_hop().estimator
    verify_inputs = phan_cum_tich_hop().verify_inputs
    from ham_dung_chung import rfm_features, segmentation_transform, read_model_parameters

    api=phan_cum_tich_hop()
    api.ensure_main_experiments(ROOT,'combined_v1')
    api.compare_models(ROOT,'combined_v1')
    api.sensitivity_models(ROOT,'combined_v1')
    api.ensure_status_sensitivity(ROOT,'combined_v1')

    class CoreTests(unittest.TestCase):
        def orders(self):
            return pd.DataFrame({'order_id':['a','b'],'customer_id':['x','x'],
                'order_date':pd.to_datetime(['2022-12-20','2022-12-20']),
                'net_amount':[Decimal('.1'),Decimal('.2')]})
        def test_money_exact_and_orders_not_days(self):
            f=rfm_features().rfm(self.orders())
            self.assertEqual(f.Monetary.iloc[0],Decimal('.3'))
            self.assertEqual(f.Frequency.iloc[0],2)
            self.assertEqual(f.Recency.iloc[0],11)
        def test_duplicate_order_rejected(self):
            x=self.orders();x.loc[1,'order_id']='a'
            with self.assertRaises(ValueError):rfm_features().rfm(x)
        def test_future_date_rejected(self):
            x=self.orders();x.loc[1,'order_date']=pd.Timestamp('2023-01-01')
            with self.assertRaises(ValueError):rfm_features().rfm(x)
        def test_negative_money_rejected(self):
            x=self.orders();x.loc[1,'net_amount']=Decimal('-1')
            with self.assertRaises(ValueError):rfm_features().rfm(x)
        def test_float_money_rejected(self):
            x=self.orders();x['net_amount']=[.1,.2]
            with self.assertRaises(ValueError):rfm_features().rfm(x)
        def test_noise_is_not_cluster(self):
            info=cluster_info(np.array([-1,-1,0,0,1]))
            self.assertEqual(info['clusters'],2)
            self.assertAlmostEqual(info['coverage'],.6)
            self.assertAlmostEqual(info['min_cluster_share_all'],.2)
        def test_single_cluster_has_no_silhouette(self):
            row=metric_row(np.array([[0.],[1.],[2.]]),np.array([0,0,0]))
            self.assertTrue(np.isnan(row['silhouette']))
        def test_log_keeps_recency_unlogged(self):
            f=pd.DataFrame({'Recency':[1,2,100],'Frequency':[1,2,30],'Monetary':[1,4,1000]})
            _,sc=segmentation_transform().feature_matrix(f,'TN2')
            self.assertAlmostEqual(sc.mean_[0],f.Recency.mean())
            self.assertAlmostEqual(sc.mean_[1],np.log1p(f.Frequency).mean())
        def test_ari_handles_label_permutation(self):
            self.assertEqual(adjusted_rand_score([0,0,1,1],[1,1,0,0]),1)
        def test_unknown_algorithm_rejected(self):
            with self.assertRaises(ValueError):estimator('unknown',{})

    class AcceptanceTests(unittest.TestCase):
        base=phan_cum_tich_hop().open_model(ROOT,'combined_v1')
        def test_sources_unchanged(self):
            self.assertEqual(len(verify_inputs(ROOT)['files']),16)
        def test_independent_raw_reconciliation(self):
            d=json.loads((ROOT/'data/processed/independent_rfm_audit.json').read_text())
            self.assertTrue(d['all_order_net_exact_match_raw'])
            self.assertTrue(d['all_customer_rfm_exact_match_baseline'])
            self.assertEqual(d['exact_net_total'],'12518175957.20')
        def test_scope_and_exact_rfm(self):
            f=pd.read_csv(self.base/'rfm_exact.csv',dtype={'Monetary':str})
            scope=pd.read_csv(self.base/'all_customer_assignments.csv')
            self.assertEqual(len(f),85115);self.assertTrue(f.customer_id.is_unique)
            self.assertEqual(int(f.Frequency.sum()),516716)
            self.assertEqual(sum(map(Decimal,f.Monetary),Decimal(0)),Decimal('12518175957.20'))
            self.assertEqual(len(scope),121930)
            self.assertEqual(scope.cluster.notna().sum(),85115)
        def test_model_parameters_reproduce_labels(self):
            f=pd.read_csv(self.base/'customer_segments.csv');s=read_model_parameters(self.base)['selected']
            x=f[['Recency','Frequency','Monetary']].to_numpy(float)
            if s['variant']=='TN2':x[:,1:]=np.log1p(x[:,1:])
            z=(x-s['scaler_mean'])/s['scaler_scale']
            centers=np.array(s['center_coordinates'])
            raw=((z[:,None,:]-centers[None,:,:])**2).sum(axis=2).argmin(axis=1)
            predicted=np.array([s['center_to_cluster'][i] for i in raw])
            np.testing.assert_array_equal(predicted,f.cluster)
        def test_36_comparisons_and_eligibility(self):
            d=pd.read_csv(self.base/'doi_chung/scores.csv')
            self.assertEqual(len(d),36)
            self.assertEqual(set(d.algorithm),{'K-Means','Hierarchical','DBSCAN','GMM'})
            eligible=pd.read_csv(self.base/'doi_chung/representatives_eligible.csv')
            self.assertTrue(eligible.eligible.all())
            self.assertTrue(eligible.min_cluster_share_all.ge(.02).all())
        def test_selected_full_seed_stability(self):
            d=pd.read_csv(self.base/'selected_full_seed_stability.csv')
            self.assertEqual(len(d),3);self.assertTrue(d.ari_full.ge(.9).all())
        def test_new_sensitivity_includes_noise_and_population(self):
            d=pd.read_csv(self.base/'doi_chung/sensitivity.csv')
            self.assertIn('ari_common_nonnoise',d)
            self.assertTrue(d.loc[d['case'].str.startswith('subset80'),'comparison_customers'].eq(4000).all())
            self.assertTrue(d.loc[d['case'].str.startswith('subset80'),'seed'].eq(42).all())
        def test_status_sensitivity_is_frozen(self):
            d=pd.read_csv(self.base/'status_sensitivity.csv')
            self.assertEqual(len(d),2)
            self.assertTrue(d.method.str.contains('Frozen').all())
        def test_final_names_and_saved_results(self):
            f=pd.read_csv(self.base/'ket_qua_phan_cum.csv')
            self.assertEqual(len(f),85115);self.assertTrue(f.customer_id.is_unique)
            self.assertEqual(f.segment.nunique(),3);self.assertFalse(f.segment.isna().any())
        def test_all_six_notebooks_really_executed(self):
            files=list((ROOT/'notebook/05_phan_khuc_khach_hang').glob('PC0*.ipynb'))
            self.assertEqual(len(files),6)
            for p in files:
                n=json.loads(p.read_text())
                self.assertTrue(n['metadata']['integration_execution']['completed'])
                for c in n['cells']:
                    if c['cell_type']=='code':
                        self.assertIsNotNone(c['execution_count'],p.name)
                        self.assertFalse(any(o['output_type']=='error' for o in c['outputs']),p.name)


    class OutputPolicyTests(unittest.TestCase):
        def study(self):
            from unittest.mock import patch
            Study = phan_cum_tich_hop().Study
            with patch.dict('os.environ', {'DATATHON_EXPORT_REPORTS':'0'}):
                return Study(ROOT,'TEST_QUIET')

        def test_default_does_not_create_report_files(self):
            from IPython.utils.capture import capture_output
            import matplotlib.pyplot as plt
            study=self.study()
            before=set(study.out.rglob('*')) if study.out.exists() else set()
            with capture_output():
                study.table('example','Ví dụ',pd.DataFrame({'x':[1]}))
                fig,ax=plt.subplots();ax.plot([1,2])
                study.fig('example','Ví dụ',fig)
                study.finish()
            after=set(study.out.rglob('*')) if study.out.exists() else set()
            self.assertEqual(before,after)
            self.assertFalse(study.export_reports)

        def test_explicit_export_remains_available(self):
            import tempfile
            from IPython.utils.capture import capture_output
            import matplotlib.pyplot as plt
            study=self.study()
            with tempfile.TemporaryDirectory() as directory:
                study.out=Path(directory)
                study.export_reports=True
                with capture_output():
                    study.table('example','Ví dụ',pd.DataFrame({'x':[1]}))
                    fig,ax=plt.subplots();ax.plot([1,2])
                    study.fig('example','Ví dụ',fig)
                    study.finish()
                self.assertTrue((study.out/'example.csv').exists())
                self.assertTrue((study.out/'example.png').exists())
                self.assertTrue((study.out/'execution.json').exists())

        def test_no_notebook_reads_report_files(self):
            import ast
            for path in (ROOT/'notebook/05_phan_khuc_khach_hang').glob('PC0*.ipynb'):
                doc=json.loads(path.read_text())
                for cell in doc['cells']:
                    if cell['cell_type']!='code':continue
                    tree=ast.parse(''.join(cell['source']))
                    for node in ast.walk(tree):
                        if isinstance(node,ast.Call) and isinstance(node.func,ast.Attribute) and node.func.attr in ['read_csv','read_text','read_bytes']:
                            self.assertNotIn('reports/',ast.unparse(node),path.name)



    class CompactStorageTests(unittest.TestCase):
        def test_old_v1_folder_removed(self):
            self.assertFalse((ROOT/'data/model/phan_khuc/v1').exists())
        def test_no_loose_intermediate_model_files(self):
            folder=ROOT/'data/model/phan_khuc/combined_v1'
            names=set(p.name for p in folder.iterdir())
            self.assertEqual(names,{'mo_hinh_phan_cum.json','ket_qua_phan_cum.csv'})
        def test_result_and_selected_model_are_complete(self):
            folder=ROOT/'data/model/phan_khuc/combined_v1'
            result=pd.read_csv(folder/'ket_qua_phan_cum.csv')
            model=json.loads((folder/'mo_hinh_phan_cum.json').read_text())
            self.assertEqual(len(result),85115)
            self.assertTrue(result.customer_id.is_unique)
            self.assertIn('center_coordinates',model['selected'])
            self.assertEqual(model['result_sha256'],phan_cum_tich_hop().sha(folder/'ket_qua_phan_cum.csv'))

    suite = unittest.TestSuite()
    for case in (CoreTests, AcceptanceTests, OutputPolicyTests, CompactStorageTests):
        suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(case))
    return unittest.TextTestRunner(verbosity=2).run(suite)



## 3. Chuẩn bị và đối soát đầu vào sau GD2

In [4]:
def chuan_bi_phan_cum():
    """Chủ động kiểm/tạo metadata cần thiết sau GD2, không xuất báo cáo phụ."""
    import json
    api = phan_cum_tich_hop()
    root = PROJECT_ROOT
    outputs = [root/'data/processed'/n for n in ('orders_common.csv','order_items_common.csv')]
    if not all(p.is_file() for p in outputs):
        raise FileNotFoundError('Chạy Run All notebook GD2 để tạo đủ hai common CSV trước.')
    target = root/'data/processed/integration_inputs.json'
    files = {p.relative_to(root).as_posix():api.sha(p)
             for p in [*sorted((root/'data/raw').glob('*.csv')),*outputs]}
    if target.exists():
        previous = json.loads(target.read_text(encoding='utf-8'))
        if previous['files'] != files:
            raise ValueError('Nguồn/common khác snapshot đã kiểm; cần nghiệm thu lượt mới, không ghi đè chứng cứ.')
    else:
        api.write_json(target, {'files':files,'method':'GD2 + independent Decimal audit','clean_files_available':False})
    return api.independent_audit(root)

